# Step 2 — Train student model (Swin3D)

Trains a **Video Swin3D-T** student directly on the 16-frame ROI clips produced in Step 1 (`temporal_extract_all_cams_16frames_roi.ipynb`). Optionally distills from a teacher checkpoint (same architecture) via KL divergence on softened logits.

**Input:** `ds_driveguard_16frames_roi.nosync/{train,val,test}/{Safe,Drink,Phone}/{sequence_id}/frame_00.jpg ... frame_15.jpg`

**Output:** `checkpoints/best_swin3d_driveguard.pt`, `checkpoints/last_swin3d_driveguard.pt`

## Setup — imports, seeding, device

In [9]:
import os
import random
from pathlib import Path
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import functional as TF
from torchvision.models.video import Swin3D_T_Weights, swin3d_t
from tqdm import tqdm


CLASS_NAMES = ["Safe", "Drink", "Phone"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}


In [10]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


## Dataset

Loads 16-frame sequences and applies identical augmentation params across all 16 frames per clip to preserve temporal consistency.

In [11]:
class DriveGuard16FramesDataset(Dataset):
    def __init__(self, root_dir: str, split: str, image_size: int = 224, is_train: bool = False) -> None:
        self.split_dir = Path(root_dir) / split
        self.image_size = image_size
        self.is_train = is_train
        self.samples: List[Tuple[List[Path], int]] = []
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )
        self.crop_scale = (0.8, 1.0)
        self.crop_ratio = (0.9, 1.1)
        self.jitter_brightness = 0.2
        self.jitter_contrast = 0.2
        self.jitter_saturation = 0.2
        self.jitter_hue = 0.05
        self._scan()

    def limit_fraction(self, fraction: float, seed: int) -> None:
        if len(self.samples) == 0:
            return
        if fraction >= 1.0:
            return
        if fraction <= 0.0:
            raise ValueError("limit_data must be in (0, 1].")
        keep_count = max(1, int(len(self.samples) * fraction))
        rng = random.Random(seed)
        chosen_indices = sorted(rng.sample(range(len(self.samples)), keep_count))
        self.samples = [self.samples[idx] for idx in chosen_indices]

    def _scan(self) -> None:
        if not self.split_dir.exists():
            return

        for class_name in CLASS_NAMES:
            class_dir = self.split_dir / class_name
            if not class_dir.exists():
                continue

            for seq_dir in class_dir.iterdir():
                if not seq_dir.is_dir():
                    continue
                frame_paths = sorted(seq_dir.glob("frame_*.jpg"))
                if len(frame_paths) == 16:
                    self.samples.append((frame_paths, CLASS_TO_IDX[class_name]))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        frame_paths, label = self.samples[idx]
        images = [Image.open(frame_path).convert("RGB") for frame_path in frame_paths]

        if self.is_train:
            # Apply identical augmentation params over all 16 frames to preserve temporal consistency.
            i, j, h, w = transforms.RandomResizedCrop.get_params(
                images[0],
                scale=self.crop_scale,
                ratio=self.crop_ratio,
            )
            do_hflip = random.random() < 0.5
            b_factor = 1.0 + random.uniform(-self.jitter_brightness, self.jitter_brightness)
            c_factor = 1.0 + random.uniform(-self.jitter_contrast, self.jitter_contrast)
            s_factor = 1.0 + random.uniform(-self.jitter_saturation, self.jitter_saturation)
            h_factor = random.uniform(-self.jitter_hue, self.jitter_hue)

            aug_images = []
            for image in images:
                image = TF.resized_crop(image, i, j, h, w, (self.image_size, self.image_size))
                if do_hflip:
                    image = TF.hflip(image)
                image = TF.adjust_brightness(image, b_factor)
                image = TF.adjust_contrast(image, c_factor)
                image = TF.adjust_saturation(image, s_factor)
                image = TF.adjust_hue(image, h_factor)
                aug_images.append(image)
            images = aug_images
        else:
            images = [TF.resize(image, (self.image_size, self.image_size)) for image in images]

        frames = [self.normalize(TF.to_tensor(image)) for image in images]

        # Swin3D expects input shape: (B, C, T, H, W)
        video = torch.stack(frames, dim=1)  # (C, T, H, W)
        return video, torch.tensor(label, dtype=torch.long)


## Model + knowledge distillation

`make_model` builds a Swin3D-T with a fresh classifier head. `load_teacher_model` loads a frozen teacher checkpoint (same architecture) for optional distillation. `kd_loss_fn` is the standard KL(student ‖ teacher) loss over softened logits, scaled by T².

In [12]:
def make_model(num_classes: int, use_pretrained: bool, dropout: float) -> nn.Module:
    if use_pretrained:
        model = swin3d_t(weights=Swin3D_T_Weights.KINETICS400_V1)
    else:
        model = swin3d_t(weights=None)
    in_features = model.head.in_features
    model.head = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),
    )
    return model


def load_teacher_model(teacher_ckpt_path: Path, device: torch.device) -> nn.Module:
    if not teacher_ckpt_path.exists():
        raise FileNotFoundError(f"Teacher checkpoint not found: {teacher_ckpt_path}")
    checkpoint = torch.load(teacher_ckpt_path, map_location=device)
    teacher_args = checkpoint.get("args", {})
    teacher_dropout = float(teacher_args.get("dropout", 0.3))
    teacher = make_model(
        num_classes=len(CLASS_NAMES),
        use_pretrained=False,
        dropout=teacher_dropout,
    ).to(device)
    teacher.load_state_dict(checkpoint["model_state_dict"])
    teacher.eval()
    for param in teacher.parameters():
        param.requires_grad = False
    return teacher


def kd_loss_fn(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    temperature: float,
) -> torch.Tensor:
    """
    KL(student || teacher) over softened class distributions.
    Multiplied by T^2 per standard distillation formulation.
    """
    log_p_student = torch.log_softmax(student_logits / temperature, dim=1)
    p_teacher = torch.softmax(teacher_logits / temperature, dim=1)
    return nn.functional.kl_div(log_p_student, p_teacher, reduction="batchmean") * (temperature ** 2)


## Evaluation + batch-size probing helpers

In [13]:
def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for videos, labels in loader:
            videos = videos.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(videos)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    if total_samples == 0:
        return 0.0, 0.0
    return total_loss / total_samples, total_correct / total_samples


def try_batch_size(
    model: nn.Module,
    criterion: nn.Module,
    sample_video: torch.Tensor,
    batch_size: int,
    device: torch.device,
    use_amp: bool,
) -> bool:
    try:
        model.zero_grad(set_to_none=True)
        videos = sample_video.unsqueeze(0).repeat(batch_size, 1, 1, 1, 1).to(device)
        labels = torch.zeros(batch_size, dtype=torch.long, device=device)
        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            logits = model(videos)
            loss = criterion(logits, labels)
        loss.backward()
        model.zero_grad(set_to_none=True)
        del videos, labels, logits, loss
        if device.type == "cuda":
            torch.cuda.empty_cache()
        if device.type == "mps":
            torch.mps.empty_cache()
        return True
    except RuntimeError as exc:
        message = str(exc).lower()
        is_memory_error = "out of memory" in message or "oom" in message
        if is_memory_error:
            if device.type == "cuda":
                torch.cuda.empty_cache()
            if device.type == "mps":
                torch.mps.empty_cache()
            return False
        raise


def auto_batch_size_probe(
    model: nn.Module,
    criterion: nn.Module,
    dataset: DriveGuard16FramesDataset,
    device: torch.device,
    use_amp: bool,
) -> None:
    if len(dataset) == 0:
        return
    sample_video, _ = dataset[0]
    candidates = [2, 4, 8]
    print("[INFO] Probing feasible batch sizes on current device...")
    feasible = []
    for candidate in candidates:
        ok = try_batch_size(model, criterion, sample_video, candidate, device, use_amp)
        print(f"[INFO] Batch size {candidate}: {'OK' if ok else 'OOM'}")
        if ok:
            feasible.append(candidate)
    if feasible:
        print(f"[INFO] Suggested max tested batch size: {max(feasible)}")
    else:
        print("[WARN] Even batch size 2 failed in probe. Lower image size may be required.")


## Training loop

Standard train/val loop with AMP, cosine LR schedule, early stopping on val loss, and optional per-step KD blending (`(1 - alpha) * CE + alpha * KD`, applied every `distill_every_n_steps` steps).

In [14]:
def train(cfg: dict) -> None:
    if cfg["quick_train"]:
        cfg["epochs"] = min(cfg["epochs"], 7)
        cfg["limit_data"] = min(cfg["limit_data"], 0.35)
        cfg["image_size"] = min(cfg["image_size"], 160)
        cfg["eval_every"] = max(cfg["eval_every"], 2)
        cfg["distill_every_n_steps"] = max(cfg["distill_every_n_steps"], 2)
        print(
            "[INFO] quick_train enabled -> "
            f"epochs={cfg['epochs']}, limit_data={cfg['limit_data']:.2f}, image_size={cfg['image_size']}, "
            f"eval_every={cfg['eval_every']}, distill_every_n_steps={cfg['distill_every_n_steps']}"
        )

    set_seed(cfg["seed"])
    device = get_device()
    print(f"[INFO] Using device: {device}")

    train_ds = DriveGuard16FramesDataset(cfg["data_root"], split="train", image_size=cfg["image_size"], is_train=True)
    val_ds = DriveGuard16FramesDataset(cfg["data_root"], split="val", image_size=cfg["image_size"], is_train=False)

    train_ds.limit_fraction(cfg["limit_data"], seed=cfg["seed"])
    val_ds.limit_fraction(cfg["limit_data"], seed=cfg["seed"] + 1)
    print(f"[INFO] Training samples after limit_data={cfg['limit_data']:.2f}: {len(train_ds)}")
    print(f"[INFO] Validation samples after limit_data={cfg['limit_data']:.2f}: {len(val_ds)}")

    if len(train_ds) == 0:
        raise RuntimeError(
            f"No training samples found in '{Path(cfg['data_root']) / 'train'}'. "
            "Run Step 1 first and verify class folders (Safe/Drink/Phone)."
        )

    num_workers = min(os.cpu_count() or 2, cfg["num_workers"])
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg["batch_size"],
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
    )

    model = make_model(num_classes=len(CLASS_NAMES), use_pretrained=cfg["pretrained"], dropout=cfg["dropout"]).to(device)
    teacher_model: Optional[nn.Module] = None
    if cfg["enable_distillation"]:
        if not cfg["teacher_checkpoint"]:
            raise ValueError("cfg['teacher_checkpoint'] is required when enable_distillation is set.")
        teacher_model = load_teacher_model(Path(cfg["teacher_checkpoint"]), device)
        print(f"[INFO] Distillation enabled. Teacher loaded from: {cfg['teacher_checkpoint']}")
        print(
            f"[INFO] Distillation params: alpha={cfg['distill_alpha']:.3f}, "
            f"temperature={cfg['distill_temperature']:.3f}"
        )
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])
    optimizer = optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(cfg["epochs"], 1),
        eta_min=cfg["min_lr"],
    )

    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    if cfg["auto_batch_test"]:
        auto_batch_size_probe(model, criterion, train_ds, device, use_amp)

    best_val_acc = -1.0
    best_val_loss = float("inf")
    epochs_without_improvement = 0
    start_epoch = 1
    os.makedirs(cfg["output_dir"], exist_ok=True)
    best_path = Path(cfg["output_dir"]) / "best_swin3d_driveguard.pt"
    last_path = Path(cfg["output_dir"]) / "last_swin3d_driveguard.pt"

    if cfg["resume_from"]:
        resume_path = Path(cfg["resume_from"])
        if not resume_path.exists():
            raise FileNotFoundError(f"Resume checkpoint not found: {resume_path}")
        checkpoint = torch.load(resume_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        if "optimizer_state_dict" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        if "scheduler_state_dict" in checkpoint:
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        start_epoch = int(checkpoint.get("epoch", 0)) + 1
        best_val_acc = float(checkpoint.get("val_acc", best_val_acc))
        best_val_loss = float(checkpoint.get("val_loss", best_val_loss))
        print(f"[INFO] Resumed from {resume_path} at epoch {start_epoch - 1}")
        if start_epoch > cfg["epochs"]:
            print("[INFO] Resume epoch already reaches/exceeds target epochs. Nothing to train.")
            return

    for epoch in range(start_epoch, cfg["epochs"] + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg['epochs']}")
        for step_idx, (videos, labels) in enumerate(pbar, start=1):
            videos = videos.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                student_logits = model(videos)
                ce_loss = criterion(student_logits, labels)

                apply_kd_this_step = (
                    teacher_model is not None
                    and (step_idx % cfg["distill_every_n_steps"] == 0)
                )

                if apply_kd_this_step:
                    with torch.no_grad():
                        teacher_logits = teacher_model(videos)
                    distill_loss = kd_loss_fn(
                        student_logits=student_logits,
                        teacher_logits=teacher_logits,
                        temperature=cfg["distill_temperature"],
                    )
                    loss = (1.0 - cfg["distill_alpha"]) * ce_loss + cfg["distill_alpha"] * distill_loss
                else:
                    distill_loss = torch.zeros((), device=videos.device)
                    loss = ce_loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * labels.size(0)
            preds = student_logits.argmax(dim=1)
            running_correct += (preds == labels).sum().item()
            total += labels.size(0)

            train_loss = running_loss / max(total, 1)
            train_acc = running_correct / max(total, 1)
            if teacher_model is not None:
                pbar.set_postfix(
                    loss=f"{train_loss:.4f}",
                    ce=f"{ce_loss.item():.4f}",
                    kd=f"{distill_loss.item():.4f}",
                    acc=f"{train_acc:.4f}",
                )
            else:
                pbar.set_postfix(loss=f"{train_loss:.4f}", acc=f"{train_acc:.4f}")

        run_eval_this_epoch = (len(val_ds) > 0) and (epoch % cfg["eval_every"] == 0)
        if run_eval_this_epoch:
            val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        elif len(val_ds) > 0:
            val_loss, val_acc = float("nan"), float("nan")
            print(f"[INFO] Epoch {epoch}: skipped validation (eval_every={cfg['eval_every']})")
        else:
            val_loss, val_acc = 0.0, 0.0
        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"[INFO] Epoch {epoch}: train_loss={running_loss / total:.4f} "
            f"train_acc={running_correct / total:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} lr={current_lr:.6f}"
        )

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_acc": val_acc,
            "val_loss": val_loss,
            "class_to_idx": CLASS_TO_IDX,
            "args": dict(cfg),
            "distillation": {
                "enabled": teacher_model is not None,
                "teacher_checkpoint": cfg["teacher_checkpoint"] if teacher_model is not None else "",
                "alpha": cfg["distill_alpha"] if teacher_model is not None else 0.0,
                "temperature": cfg["distill_temperature"] if teacher_model is not None else 0.0,
            },
        }
        torch.save(checkpoint, last_path)

        if len(val_ds) > 0 and run_eval_this_epoch:
            improved = val_loss < (best_val_loss - cfg["early_stop_min_delta"])
        elif len(val_ds) == 0:
            improved = val_acc >= best_val_acc
        else:
            improved = False

        if improved:
            best_val_acc = val_acc
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save(checkpoint, best_path)
            print(f"[INFO] Saved new best model to {best_path}")
        elif len(val_ds) > 0 and run_eval_this_epoch:
            epochs_without_improvement += 1
            if epochs_without_improvement >= cfg["early_stopping_patience"]:
                print(
                    f"[INFO] Early stopping triggered after {epoch} epochs "
                    f"(no val_loss improvement for {cfg['early_stopping_patience']} epochs)."
                )
                break

        scheduler.step()

    print(f"[INFO] Training complete. Last model: {last_path}")
    print(f"[INFO] Best model: {best_path} (val_acc={best_val_acc:.4f})")


## Config — edit these, then run training

Mirrors the original `train_model.py` CLI defaults. Set `enable_distillation=True` and `teacher_checkpoint` to distill from a teacher checkpoint of the same architecture.

In [15]:
cfg = {
    "data_root": "ds_driveguard_16frames_roi.nosync",
    "output_dir": "checkpoints",
    "epochs": 8,
    "batch_size": 2,
    "limit_data": 0.3,
    "resume_from": "",
    "auto_batch_test": False,
    "lr": 1e-4,
    "min_lr": 1e-6,
    "weight_decay": 1e-4,
    "dropout": 0.3,
    "label_smoothing": 0.1,
    "enable_distillation": False,
    "teacher_checkpoint": "",
    "distill_alpha": 0.5,
    "distill_temperature": 2.0,
    "distill_every_n_steps": 1,
    "eval_every": 1,
    "quick_train": False,
    "early_stopping_patience": 3,
    "early_stop_min_delta": 1e-4,
    "image_size": 224,
    "num_workers": 0,
    "seed": 42,
    "pretrained": True,
}


In [16]:
train(cfg)


[INFO] Using device: mps
[INFO] Training samples after limit_data=0.30: 3077
[INFO] Validation samples after limit_data=0.30: 813


Epoch 1/8: 100%|██████████| 1539/1539 [19:15<00:00,  1.33it/s, acc=0.5509, loss=0.9793]


[INFO] Epoch 1: train_loss=0.9793 train_acc=0.5509 val_loss=0.1702 val_acc=0.0000 lr=0.000100
[INFO] Saved new best model to checkpoints/best_swin3d_driveguard.pt


Epoch 2/8: 100%|██████████| 1539/1539 [19:32<00:00,  1.31it/s, acc=0.7173, loss=0.7783]


[INFO] Epoch 2: train_loss=0.7783 train_acc=0.7173 val_loss=0.1600 val_acc=0.0000 lr=0.000096
[INFO] Saved new best model to checkpoints/best_swin3d_driveguard.pt


...

[INFO] Epoch 3: train_loss=0.6345 train_acc=0.8102 val_loss=0.1997 val_acc=0.0000 lr=0.000086


Epoch 4/8: 100%|██████████| 1539/1539 [19:01<00:00,  1.35it/s, acc=0.8560, loss=0.5583]


[INFO] Epoch 4: train_loss=0.5583 train_acc=0.8560 val_loss=0.1816 val_acc=0.0000 lr=0.000069


Epoch 5/8: 100%|██████████| 1539/1539 [17:40<00:00,  1.45it/s, acc=0.9074, loss=0.4655]


[INFO] Epoch 5: train_loss=0.4655 train_acc=0.9074 val_loss=0.2093 val_acc=0.0000 lr=0.000051
[INFO] Early stopping triggered after 5 epochs (no val_loss improvement for 3 epochs).
[INFO] Training complete. Last model: checkpoints/last_swin3d_driveguard.pt
[INFO] Best model: checkpoints/best_swin3d_driveguard.pt (val_acc=0.0000)
